# Data Collection and Preparation - Health Emergencies Preparedness and Response Program (HEPR)

This notebook is a template workflow to collect data and prepare the main data to perform a baseline physical accessibility analysis to health facilities. It uses various tools developed by the World Bank's Geospatial Operations Support Team (GOST).

This notebook focuses on a vector-based implementation of market access, using a road network from Open Street Map. Additionaly, it uses population data from [World Pop](https://hub.worldpop.org/project/categories?id=3) (Unconstrained UN-Adjusted 2020, 1km resolution).

## Data Download Links
- [World Pop Raster](https://data.worldpop.org/GIS/Population/Global_2000_2020_1km/2020/)
- [Open Street Map Road Network](https://download.geofabrik.de/)

## 0.Setup

Import packages required for the analysis

In [ ]:
import sys
from os.path import join, expanduser
import geopandas as gpd
import pandas as pd
from gadm import GADMDownloader
from tqdm import tqdm

In [ ]:
# Visualization tools
# import folium as flm
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from rasterio.plot import plotting_extent
from rasterio.plot import show
from mpl_toolkits.axes_grid1 import make_axes_locatable
import contextily as ctx


In [ ]:
# Raster
import rasterio as rio
import numpy as np
from shapely.geometry import Polygon, box, Point
import skimage.graph as graph

# Graph
import pickle
import networkx as nx
import osmnx as ox

In [ ]:
# Define your path to the Repositories

sys.path.append(join(expanduser("~/World_Bank"), 'Repos', 'gostrocks', 'src'))
sys.path.append(join(expanduser("~/World_Bank"), 'Repos', 'GOSTNets_Raster', 'src'))
sys.path.append(join(expanduser("~/World_Bank"), 'Repos', 'GOSTnets'))
sys.path.append(join(expanduser("~/World_Bank"), 'Repos', 'GOST_Urban', 'src', 'GOST_Urban'))

import GOSTnets as gn
from GOSTnets.load_osm import *
import GOSTRocks.rasterMisc as rMisc
from GOSTRocks.misc import get_utm
import GOSTNetsRaster.market_access as ma
import UrbanRaster as urban

In [ ]:
# auto reload
%load_ext autoreload
%autoreload 2

## 1. Data Preparation


In [ ]:
scratch_dir = join(expanduser("~/World_Bank"), 'Health-Access-Metrics', 'data')
scratch_dir

In [ ]:
# Example of Pilot country

country = 'Mozambique'
iso = 'mzb'
downloader = GADMDownloader(version="4.0")
adm3 = downloader.get_shape_data_by_country_name(country_name=country, ad_level=3)
assert isinstance(adm3, gpd.GeoDataFrame)
adm3.plot()

### Origins

Import Population data from WorldPop database for the pilot country of interest 

In [ ]:
wp_path = join('scratch_dir', 'data', 'worldpop', f'{iso}_ppp_2020_1km_Aggregated.tif')
wp_rio = rio.open(wp_path)
pop = wp_rio.read(1, masked=True)

In [ ]:
fig, ax = plt.subplots(figsize=(6,8))
im = ax.imshow(pop, norm=colors.PowerNorm(gamma=0.5), cmap='viridis')

### Origins

We prepare a standard grid using each cell from the 1km World Pop raster.

In [ ]:
indices = list(np.ndindex(pop.shape))
xys = [Point(wp_rio.xy(ind[0], ind[1])) for ind in indices]
res_df = pd.DataFrame({
    'spatial_index': indices, 
    'xy': xys, 
    'pop': pop.flatten()
})
res_df['pointid'] = res_df.index

In [ ]:
res_df = res_df.loc[res_df['pop']>0].copy()
res_df = res_df.loc[~(res_df['pop'].isna())].copy()     # ~ symbol returns the opposite of Boolean
res_df.reset_index(inplace=True, drop=True)
origins = gpd.GeoDataFrame(res_df, geometry='xy', crs='EPSG:4326')
origins.rename(columns={'xy':'geometry'}, inplace=True)
origins.set_geometry('geometry', inplace=True)

### Destinations (Cities)

In this section, we create a geodataframe of cities based on the population data. We follow the [European Comission's Degree of Urbanization methodology](https://ghsl.jrc.ec.europa.eu/degurba.php), which defines urban areas as clusters of 5,000 people with a population density greater than 300 people per sq. km. 

The code below uses the World Pop raster to identify the clusters (contiguous cells that meet the thresholds).

In [ ]:
urban_calculator = urban.urbanGriddedPop(wp_path)

In [ ]:
urban_extents = urban_calculator.calculateUrban(
    densVal=190, totalPopThresh=5000,  # changed the density value to capute more remote cities
    smooth=True, queen=False, verbose=True
    )

In [ ]:
len(urban_extents)

In [ ]:
# urban_extents.explore()

Convert the urban extents to points using the centroid.

In [ ]:
dests = urban_extents.copy()
dests.loc[:, 'geometry'] = dests.geometry.centroid

### Road Network

In [ ]:
minx, miny, maxx, maxy = adm0.total_bounds

Here we load Open Street Map road network as a networkx graph. I tested both OSMNX and the traditional GOSTnets approach. 

#### Retrieve using OSMNX

In [ ]:
import psutil

# Get the current process's memory usage
process = psutil.Process(os.getpid())
memory_info = process.memory_info()

# Print the memory usage
print(f"Memory usage: {memory_info.rss / (1024 ** 2):.2f} MB")


# %%time
G1 = ox.graph_from_bbox((maxy - miny)/3 + miny, miny, (maxx - minx)/3 + minx, minx, network_type='all', retain_all=True)

In [ ]:
def download_and_union_graphs(xmin, xmax, ymin, ymax, n):
    # Create an empty graph for union
    final_graph = nx.MultiDiGraph()

    # Calculate the step size for splitting the bounding box
    x_step = (xmax - xmin) / n
    y_step = (ymax - ymin) / n

    # Iterate over rows
    for i in range(n):
        x1 = xmin + i * x_step
        x2 = x1 + x_step

        # Iterate over columns
        for j in range(n):
            y1 = ymin + j * y_step
            y2 = y1 + y_step

            # Download graph data for the current bounding box
            current_graph = ox.graph_from_bbox(y2, y1, x2, x1, network_type='all', retain_all=True)
            
            print(f"Downloaded sub-box {i+1}-{j+1}: ({x1},{y1}) to ({x2},{y2})")

            # Union the current graph with the final graph
            final_graph = nx.union(final_graph, current_graph)

    return final_graph


In [ ]:

n = 3  # Specify the number of smaller boxes in each dimension

G = nx.MultiDiGraph()

# Calculate the step size for splitting the bounding box
x_step = (maxx - minx) / n
y_step = (maxy - miny) / n

# Iterate over rows
for i in range(n):
    x1 = minx + i * x_step
    x2 = x1 + x_step

# Iterate over columns
    for j in range(n):
        y1 = miny + j * y_step
        y2 = y1 + y_step

# Download graph data for the current bounding box
        current_graph = ox.graph_from_bbox(y2, y1, x2, x1, network_type='all', retain_all=True)

        print(f"Downloaded sub-box {i+1}-{j+1}: ({x1},{y1}) to ({x2},{y2})")

# Union the current graph with the final graph
        G = nx.union(G, current_graph)

# By downloading data iteratively
# G = download_osm_data_iteratively(minx, maxx, miny, maxy, 3)


In [ ]:
fig, ax = ox.plot_graph(G)

In [ ]:
gn.save(G, f"G_{iso}", scratch_dir, pickle=False) # you can save edegs and nodes as csv
with open(join(scratch_dir, f'G_{iso}_osmnx_all.gpickle'), 'wb') as f:
    pickle.dump(G, f, pickle.HIGHEST_PROTOCOL)

Load Network

In [ ]:
with open(join(scratch_dir, f'G_{iso}_osmnx_all.gpickle'), 'rb') as f:
    G = pickle.load(f)

In [ ]:
edges = gn.edge_gdf_from_graph(G)

### Destinations (Health Facilities)

### Road network

In [ ]:
minx, miny, maxx, maxy = adm3.total_bounds

#### Retrieve using OSMNX

In [ ]:
# Function for downloading subboxes of the geographical domain

def download_and_union_graphs(xmin, xmax, ymin, ymax, n):
    # Create an empty graph for union
    final_graph = nx.MultiDiGraph()

    # Calculate the step size for splitting the bounding box
    x_step = (xmax - xmin) / n
    y_step = (ymax - ymin) / n

    # Iterate over rows
    for i in range(n):
        x1 = xmin + i * x_step
        x2 = x1 + x_step

        # Iterate over columns
        for j in range(n):
            y1 = ymin + j * y_step
            y2 = y1 + y_step

            # Download graph data for the current bounding box
            current_graph = ox.graph_from_bbox(y2, y1, x2, x1, network_type='all', retain_all=True)
            
            print(f"Downloaded sub-box {i} of {n*n}")

            # Union the current graph with the final graph
            final_graph = nx.union(final_graph, current_graph)

    return final_graph

G = download_osm_data_iteratively(minx, maxx, miny, maxy, 3)


In [ ]:
fig, ax = ox.plot_graph(G)

In [ ]:
# Save the MultiDiGraph object

gn.save(G, f"G_{iso}", scratch_dir, pickle=False) # you can save edges and nodes as csv
with open(join(scratch_dir, f'G_{iso}_osmnx_all.gpickle'), 'wb') as f:
    pickle.dump(G, f, pickle.HIGHEST_PROTOCOL)

Load Network

In [ ]:
with open(join(scratch_dir, f'G_{iso}_osmnx_all.gpickle'), 'rb') as f:
    G = pickle.load(f)

In [ ]:
edges = gn.edge_gdf_from_graph(G)

In [ ]:
edges.highway.value_counts()

In [ ]:
list_of_subgraphs = [G.subgraph(c).copy() for c in sorted(nx.strongly_connected_components(G), key=len, reverse=True)]
# G_largest = list_of_subgraphs[0]

In [ ]:
len(list_of_subgraphs)

In [ ]:
edges = gn.edge_gdf_from_graph(G)

In [ ]:
edges.highway.value_counts()

In [ ]:
list_of_subgraphs = [G.subgraph(c).copy() for c in sorted(nx.strongly_connected_components(G), key=len, reverse=True)]
# G_largest = list_of_subgraphs[0]

In [ ]:
len(list_of_subgraphs)

#### Retrieve using GOSTnets
For now, I am using the GOSTnets implementation as it includes ferries and the classification of roads is a bit cleaner.

In [ ]:
osm_pbf = join(expanduser("~/World_Bank"), 'data', 'xxxxxx.osm.pbf')

In [ ]:
osm_raw = OSM_to_network(osm_pbf, includeFerries=True)

In [ ]:
osm_raw.roads_raw.infra_type.value_counts()

Remove a few types. We could also remove 'path' but it seemed like there were a lot of roads tagged as 'path' that were actually roads.

In [ ]:
unaccepted_road_types = ['footway', 'steps', 'corridor', 'pedestrian']
accepted_road_types = list(osm_raw.roads_raw.infra_type.unique())

In [ ]:
accepted_road_types = [road_type for road_type in accepted_road_types if road_type not in unaccepted_road_types]

In [ ]:
osm_raw.filterRoads(acceptedRoads = accepted_road_types)

In [ ]:
osm_raw.generateRoadsGDF(verbose = False)
osm_raw.initialReadIn()

In [ ]:
osm_raw

Get UTM Zone, GOSTnets projects edges for some distance calculations.

In [ ]:
utm = get_utm(adm0)
utm = utm.to_string()
utm

This function cleans the network by removing excessive nodes, and ensures all edges are bi-directional (except in the case of one-way roads)

In [ ]:
G_clean = gn.clean_network(osm_raw.network, UTM = utm, WGS = "4326", junctdist= 50, verbose = False)

In [ ]:
G_clean.graph['crs'] = '4326'

In [ ]:
# gn.save(G, f"G_{iso}", scratch_dir, pickle=False) # you can save edegs and nodes as csv
with open(join(scratch_dir, f'G_{iso}_clean.gpickle'), 'wb') as f:
    pickle.dump(G_clean, f, pickle.HIGHEST_PROTOCOL)

Load GN network

In [ ]:
with open(join(scratch_dir, f'G_{iso}_clean.gpickle'), 'rb') as f:
    G_clean = pickle.load(f)